In [33]:
# ============================================================
# CELL 1 - Imports, State definition, and LLM initialization
# ============================================================

import os
from dotenv import load_dotenv
from typing import Annotated

from langgraph.graph.message import add_messages
from typing_extensions import TypedDict

from langchain_google_genai import ChatGoogleGenerativeAI

# Load environment variables from .env file
load_dotenv(override=True)

# ---- State definition ----
# This TypedDict defines the shared memory that travels between all agents.
# Every key added here will be accessible and writable by each node in the graph.

class PipelineState(TypedDict):
    # Accumulates all messages exchanged across agents (never overwritten, only appended)
    messages: Annotated[list, add_messages]
    # Will hold the list of suspicious transactions found by the Data Agent
    suspicious_records: list
    # Will hold the final risk report produced by the Risk Agent
    risk_report: str

# ---- LLM initialization ----
# We use gemini-1.5-flash: fast, cost-effective, sufficient for structured agent reasoning

llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0,  # Deterministic output - critical for data analysis tasks
)

# Sanity check
print("LLM initialized:", llm.model)
print("State keys defined:", list(PipelineState.__annotations__.keys()))
print("LLM ready with key:", os.getenv("GOOGLE_API_KEY")[:8], "...")

LLM initialized: gemini-2.0-flash
State keys defined: ['messages', 'suspicious_records', 'risk_report']
LLM ready with key: AIzaSyAp ...


In [34]:
# ============================================================
# CELL 2 - Load and inspect the two Excel files
# ============================================================

import pandas as pd

# Go one level up from the current working directory
PATH_ALLARMI    = r"C:\Users\gmitr\OneDrive\Documents\ML Project Reply\ALLARMI.xlsx"
PATH_TIPOLOGIA  = r"C:\Users\gmitr\OneDrive\Documents\ML Project Reply\TIPOLOGIA_VIAGGIATORE.xlsx"

df_allarmi   = pd.read_excel(PATH_ALLARMI)
df_tipologia = pd.read_excel(PATH_TIPOLOGIA)

print("=== ALLARMI ===")
print(f"Shape: {df_allarmi.shape}")
print(df_allarmi.dtypes)
print()
print("=== TIPOLOGIA_VIAGGIATORE ===")
print(f"Shape: {df_tipologia.shape}")
print(df_tipologia.dtypes)

=== ALLARMI ===
Shape: (5080, 25)
OCCORRENZE                  str
AREOPORTO_ARRIVO            str
AREOPORTO_PARTENZA          str
ANNO_PARTENZA            object
MESE_PARTENZA            object
DATA_PARTENZA            object
DESCR_AEREOPORTO_ARR     object
DESCR_AEREOPORTO_PART       str
CITTA_ARR                   str
CITTA_PARTENZA              str
CODICE_PAESE_ARR            str
CODICE_PAESE_PART        object
PAESE_ARR                   str
PAESE_PART                  str
ZONA                     object
TOT                      object
MOTIVO_ALLARME           object
note_operatore              str
flag_rischio                str
Paese Partenza              str
CODICE PAESE ARR            str
3zona                    object
paese%arr                object
tot voli                 object
Unnamed: 24              object
dtype: object

=== TIPOLOGIA_VIAGGIATORE ===
Shape: (5095, 33)
NAZIONALITA                 str
AREOPORTO_ARRIVO            str
AREOPORTO_PARTENZA          str
ANNO_PA

In [35]:
# ============================================================
# CELL 3 - Define the LangGraph State with correct keys
# ============================================================

from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages

class PipelineState(TypedDict):
    # Message history across all agents (never overwritten, only appended)
    messages: Annotated[list, add_messages]

    # Agent 1 output: cleaned and merged DataFrame as JSON string
    # We use JSON string and never pass the raw DataFrame to avoid token explosion
    cleaned_data_json: str

    # Agent 2 output: baseline statistics per airport (compact dict as string)
    baseline_stats: str

    # Agent 3 output: list of anomalous records (only the flagged rows, not all 5000)
    anomalies: list

    # Agent 4 output: anomalies that also triggered at least one business rule
    confirmed_anomalies: list

    # Agent 5 output: final natural language report
    final_report: str

print("State keys:", list(PipelineState.__annotations__.keys()))

State keys: ['messages', 'cleaned_data_json', 'baseline_stats', 'anomalies', 'confirmed_anomalies', 'final_report']


In [36]:
# ============================================================
# CELL 4a - File paths and column rename hints only
# ============================================================

PATH_ALLARMI   = r"C:\Users\gmitr\OneDrive\Documents\ML Project Reply\ALLARMI.xlsx"
PATH_TIPOLOGIA = r"C:\Users\gmitr\OneDrive\Documents\ML Project Reply\TIPOLOGIA_VIAGGIATORE.xlsx"

# These are rename hints for columns we can identify by name.
# They are NOT a whitelist. The agent loads ALL columns and decides
# itself which ones to keep or drop based on content quality.

ALLARMI_RENAME = {
    "OCCORRENZE":            "occurrences",
    "AREOPORTO_ARRIVO":      "arrival_airport_iata",
    "AREOPORTO_PARTENZA":    "departure_airport_iata",
    "ANNO_PARTENZA":         "departure_year",
    "MESE_PARTENZA":         "departure_month",
    "DATA_PARTENZA":         "departure_date",
    "DESCR_AEREOPORTO_ARR":  "arrival_airport_name",
    "DESCR_AEREOPORTO_PART": "departure_airport_name",
    "CITTA_ARR":             "arrival_city",
    "CITTA_PARTENZA":        "departure_city",
    "CODICE_PAESE_ARR":      "arrival_country_code",
    "CODICE_PAESE_PART":     "departure_country_code",
    "PAESE_ARR":             "arrival_country",
    "PAESE_PART":            "departure_country",
    "ZONA":                  "risk_zone",
    "TOT":                   "total_flights",
    "MOTIVO_ALLARME":        "alarm_reason",
    "flag_rischio":          "risk_flag",
}

TIPOLOGIA_RENAME = {
    "NAZIONALITA":           "nationality",
    "AREOPORTO_ARRIVO":      "arrival_airport_iata",
    "AREOPORTO_PARTENZA":    "departure_airport_iata",
    "ANNO_PARTENZA":         "departure_year",
    "MESE_PARTENZA":         "departure_month",
    "GIORNO_PARTENZA":       "departure_day",
    "DATA_PARTENZA":         "departure_date",
    "DESCR_AEREOPORTO_ARR":  "arrival_airport_name",
    "DESCR_AEREOPORTO_PART": "departure_airport_name",
    "CITTA_ARR":             "arrival_city",
    "CITTA_PARTENZA":        "departure_city",
    "CODICE_PAESE_ARR":      "arrival_country_code",
    "CODICE_PAESE_PART":     "departure_country_code",
    "PAESE_ARR":             "arrival_country",
    "PAESE_PART":            "departure_country",
    "ZONA":                  "risk_zone",
    "ENTRATI":               "entries",
    "INVESTIGATI":           "investigated",
    "ALLARMATI":             "alarmed",
    "TIPO_DOCUMENTO":        "document_type",
    "GENERE":                "gender",
    "FASCIA_ETA":            "age_group",
    "FLAG_TRANSITO":         "transit_flag",
    "COMPAGNIA_AEREA":       "airline",
    "NUMERO_VOLO":           "flight_number",
    "ESITO_CONTROLLO":       "control_outcome",
    "codice_rischio":        "risk_code",
}

# Merge keys expected after renaming
MERGE_KEYS = ["departure_airport_iata", "arrival_airport_iata", "departure_date"]

print("Configuration loaded.")
print("  Rename hints defined. Agent will load ALL columns and audit them.")
print(f"  ALLARMI   known columns : {len(ALLARMI_RENAME)}")
print(f"  TIPOLOGIA known columns : {len(TIPOLOGIA_RENAME)}")

Configuration loaded.
  Rename hints defined. Agent will load ALL columns and audit them.
  ALLARMI   known columns : 18
  TIPOLOGIA known columns : 27


In [37]:
# ============================================================
# CELL 4b - Data Agent system prompt (final version)
# ============================================================

DATA_AGENT_PROMPT = """
You are an expert Data Engineer specializing in messy real-world datasets.
Your mission is to load, audit, clean, and merge two raw Excel files containing
airport border control transit data, and produce a single analysis-ready DataFrame.

You work autonomously. You do not ask for instructions on how to clean the data.
You apply the following quality standards to every dataset you receive.

STEP 1 - STRUCTURAL AUDIT
Load both files and inspect shape, column names, and data types.
Identify junk columns: Unnamed columns, columns whose name starts with a digit
or a special character, and columns with near-zero variance.
Report your findings before taking any action.

STEP 2 - DUPLICATE COLUMN RESOLUTION
Do not rely on column names alone to detect duplicates. Compare column contents.
For every pair of columns that appear to represent the same concept:
- Compute the missing rate of each.
- Compute the rate of valid values (non-null, non-negative for numerics,
  non-empty for strings) of each.
- Keep the column with the better content quality score.
- Drop the weaker column and document your choice with the quality scores
  that justified it.

STEP 3 - DATE INTEGRITY CHECK
For every date-related column, check consistency across all date representations.
If a derived column such as month or year is inconsistent with its source date
column, always trust the source date column and recompute derived values from it.
Document every inconsistency you find with the number of affected rows.

STEP 4 - MISSING VALUE ANALYSIS
Compute the missing rate for every column and apply this decision rule:
- Missing rate above 60 percent : drop the column entirely.
- Missing rate between 20 and 60 percent : keep, fill with UNKNOWN for
  categorical columns or median for numeric columns, and flag the column.
- Missing rate below 20 percent : fill with mode for categorical columns
  or median for numeric columns.
Document every decision with the exact number of rows affected.

STEP 5 - CARDINALITY AND SEMANTIC NORMALIZATION
For every categorical column apply these rules in order:

Format standardization first: strip whitespace, normalize accents,
convert to uppercase. Do this before any other check.

Redundancy detection: if a column contains multiple representations of the
same concept after format standardization, merge them into one canonical value.
- Gender variants: H, M, MALE, HOMME, MONSIEUR become MALE.
  F, FEMALE, FEMME, MADAME become FEMALE.
- Boolean variants: OUI, YES, 1, TRUE become YES. NON, NO, 0, FALSE become NO.
For any other column, use your judgment to detect semantic duplicates
and document the mapping you applied.

Zero variance after normalization: if a column has only one unique value,
drop it and document why.

Free-text detection: if a column has more unique values than 90 percent of
the row count, flag it as free-text and keep it without normalizing.

STEP 6 - BUSINESS CONSTRAINT VALIDATION
- Count columns must be non-negative. Replace negatives with zero.
- Rate columns must be between 0 and 100. Clip values outside this range.
- IATA codes must be exactly 3 uppercase letters. Flag violations.
- Departure date must not be in the future. Flag violations.
Log the number of values corrected for each rule.

STEP 7 - MERGE
Merge the two cleaned DataFrames on the shared key columns.
Report rows matched and rows lost. If more than 20 percent of rows are lost,
raise a WARNING and explain the likely cause.

STEP 8 - TECHNICAL LOG
Produce a concise technical log of at most 15 lines summarizing every decision
made in steps 1 through 7 with counts. This log is for the next agents,
not for the final report. Keep it factual and short.
Your final message must end with exactly this line:
CLEANING_COMPLETE: <number_of_rows> rows, <number_of_columns> columns
"""

print("System prompt defined.")
print(f"Prompt length: {len(DATA_AGENT_PROMPT)} characters")

System prompt defined.
Prompt length: 3810 characters


In [38]:
# ============================================================
# CELL 4c - Tool 1 : load_raw_data
# ============================================================

import pandas as pd
import numpy as np
from langchain_core.tools import tool

# Module-level storage (never passed to the LLM, only between tools)
raw_data = {}

@tool
def load_raw_data(instruction: str) -> str:
    """
    Loads both Excel files in full (all columns, no filtering).
    Returns a structural audit report for the LLM to reason about.
    The raw DataFrames are stored in memory for the next tool.
    Never returns the actual data rows to avoid token explosion.
    """

    global raw_data

    # ---- Load everything, no column filtering ----
    df_a = pd.read_excel(PATH_ALLARMI)
    df_t = pd.read_excel(PATH_TIPOLOGIA)

    raw_data["allarmi"]   = df_a
    raw_data["tipologia"] = df_t

    # ---- Build audit report ----
    def audit(df, name):
        lines = [f"=== {name} : {df.shape[0]} rows x {df.shape[1]} columns ==="]

        # Column names, types, missing rate, unique count
        lines.append(f"{'COLUMN':<30} {'TYPE':<12} {'MISSING%':<12} {'UNIQUE'}")
        lines.append("-" * 65)
        for col in df.columns:
            missing_pct = round(df[col].isna().mean() * 100, 1)
            unique_count = df[col].nunique(dropna=True)
            lines.append(f"{col:<30} {str(df[col].dtype):<12} {missing_pct:<12} {unique_count}")

        # Sample of 2 rows to show actual values
        lines.append(f"\nSample (2 rows):")
        lines.append(df.head(2).to_string())
        return "\n".join(lines)

    report = audit(df_a, "ALLARMI") + "\n\n" + audit(df_t, "TIPOLOGIA_VIAGGIATORE")

    return report


print("Tool registered:", load_raw_data.name)

Tool registered: load_raw_data


In [39]:
# ============================================================
# CELL 4d - Tool 2 : run_cleaning_code
# ============================================================

import pandas as pd
import numpy as np
from langchain_core.tools import tool

cleaned_data = {}

@tool
def run_cleaning_code(decisions: str) -> str:
    """
    Executes cleaning operations on the raw DataFrames based on the
    decisions YOU made after auditing the data with load_raw_data.

    Pass your decisions using this exact format:
    DROP_COLUMNS_ALLARMI: col1, col2
    DROP_COLUMNS_TIPOLOGIA: col1, col2
    GENDER_MAP: observed_val1->MALE, observed_val2->FEMALE
    BOOLEAN_MAP: observed_val1->YES, observed_val2->NO

    Only include lines relevant to what you actually observed.
    Do not invent values. Use only what the audit report showed you.
    """

    global cleaned_data

    df_a = raw_data["allarmi"].copy()
    df_t = raw_data["tipologia"].copy()

    log = []

    # ---- Parse the LLM decisions from the structured string ----
    def parse_list(key, text):
        for line in text.splitlines():
            if line.strip().startswith(key + ":"):
                values = line.split(":", 1)[1].strip()
                return [v.strip() for v in values.split(",") if v.strip()]
        return []

    def parse_map(key, text):
        result = {}
        for line in text.splitlines():
            if line.strip().startswith(key + ":"):
                pairs = line.split(":", 1)[1].strip()
                for pair in pairs.split(","):
                    if "->" in pair:
                        src, tgt = pair.split("->")
                        result[src.strip().upper()] = tgt.strip().upper()
        return result

    # All values come from the LLM decisions string, never hardcoded here
    drop_allarmi   = parse_list("DROP_COLUMNS_ALLARMI",   decisions)
    drop_tipologia = parse_list("DROP_COLUMNS_TIPOLOGIA",  decisions)
    gender_map     = parse_map("GENDER_MAP",               decisions)
    boolean_map    = parse_map("BOOLEAN_MAP",              decisions)

    # ---- Apply rename hints ----
    df_a = df_a.rename(columns={k: v for k, v in ALLARMI_RENAME.items()   if k in df_a.columns})
    df_t = df_t.rename(columns={k: v for k, v in TIPOLOGIA_RENAME.items() if k in df_t.columns})
    log.append(f"Columns renamed using hints")

    # ---- Drop columns the LLM decided to remove ----
    dropped_a = [c for c in drop_allarmi   if c in df_a.columns]
    dropped_t = [c for c in drop_tipologia if c in df_t.columns]
    df_a = df_a.drop(columns=dropped_a)
    df_t = df_t.drop(columns=dropped_t)
    log.append(f"Dropped ALLARMI   : {dropped_a if dropped_a else 'none'}")
    log.append(f"Dropped TIPOLOGIA : {dropped_t if dropped_t else 'none'}")

    # ---- Parse dates and recompute derived columns from source ----
    for df in [df_a, df_t]:
        if "departure_date" in df.columns:
            df["departure_date"]  = pd.to_datetime(
                df["departure_date"], errors="coerce", dayfirst=True
            )
            df["departure_year"]  = df["departure_date"].dt.year
            df["departure_month"] = df["departure_date"].dt.month
    log.append("departure_year and departure_month recomputed from departure_date")

    # ---- Fix numeric types and enforce non-negative business constraint ----
    for col in ["occurrences", "total_flights"]:
        if col in df_a.columns:
            before = (df_a[col] < 0).sum()
            df_a[col] = pd.to_numeric(df_a[col], errors="coerce").clip(lower=0)
            log.append(f"{col}: {before} negative values clipped to 0")

    for col in ["entries", "investigated", "alarmed"]:
        if col in df_t.columns:
            before = (pd.to_numeric(df_t[col], errors="coerce") < 0).sum()
            df_t[col] = pd.to_numeric(df_t[col], errors="coerce").clip(lower=0)
            log.append(f"{col}: {before} negative values clipped to 0")

    # ---- Fill missing values ----
    for df in [df_a, df_t]:
        for col in df.columns:
            missing = df[col].isna().sum()
            if missing > 0:
                if df[col].dtype == object:
                    df[col] = df[col].fillna("UNKNOWN")
                else:
                    df[col] = df[col].fillna(df[col].median())
    log.append("Missing values filled: UNKNOWN for categorical, median for numeric")

    # ---- Apply semantic normalization maps decided by the LLM ----
    if "gender" in df_t.columns and gender_map:
        df_t["gender"] = (
            df_t["gender"].astype(str).str.strip().str.upper()
            .replace(gender_map)
        )
        log.append(f"Gender normalized: {gender_map}")

    if boolean_map:
        for df in [df_a, df_t]:
            for col in df.columns:
                if df[col].dtype == object:
                    df[col] = (
                        df[col].astype(str).str.strip().str.upper()
                        .replace(boolean_map)
                    )
        log.append(f"Boolean values normalized: {boolean_map}")

    # ---- Store cleaned frames for tool 3 ----
    cleaned_data["allarmi"]   = df_a
    cleaned_data["tipologia"] = df_t

    log.append(f"ALLARMI after cleaning   : {df_a.shape}")
    log.append(f"TIPOLOGIA after cleaning : {df_t.shape}")

    return "\n".join(log)


print("Tool registered:", run_cleaning_code.name)

Tool registered: run_cleaning_code


In [40]:
# ============================================================
# CELL 4e - Tool 3 : store_clean_data
# ============================================================

from langchain_core.tools import tool
import json

@tool
def store_clean_data(instruction: str) -> str:
    """
    Call this tool only when you are satisfied with the cleaning result.
    It merges the two cleaned DataFrames on the shared key columns,
    stores the result in the pipeline State, and returns a final
    confirmation summary.

    The full dataset is never returned to avoid token explosion.
    Only call this after run_cleaning_code has been executed.
    """

    df_a = cleaned_data["allarmi"]
    df_t = cleaned_data["tipologia"]

    # ---- Merge on shared keys ----
    available_keys = [k for k in MERGE_KEYS if k in df_a.columns and k in df_t.columns]
    df_master = pd.merge(df_a, df_t, on=available_keys, how="inner",
                         suffixes=("_alarm", "_traveler"))

    rows_a      = len(df_a)
    rows_t      = len(df_t)
    rows_merged = len(df_master)
    loss_pct    = round((1 - rows_merged / min(rows_a, rows_t)) * 100, 1)

    # ---- Warn if too many rows lost ----
    merge_warning = ""
    if loss_pct > 20:
        merge_warning = f"WARNING: {loss_pct}% of rows lost in merge. Check key columns."

    # ---- Store as JSON in the global pipeline state ----
    # Only the JSON goes to State, never to the LLM context
    store_clean_data._cleaned_json = df_master.to_json(
        orient="records", date_format="iso"
    )
    store_clean_data._cleaned_columns = list(df_master.columns)
    store_clean_data._cleaned_shape   = df_master.shape

    summary = (
        f"Merge complete on keys: {available_keys}\n"
        f"  ALLARMI rows        : {rows_a}\n"
        f"  TIPOLOGIA rows      : {rows_t}\n"
        f"  Merged rows         : {rows_merged}\n"
        f"  Row loss            : {loss_pct}%\n"
        f"  Final columns       : {list(df_master.columns)}\n"
        f"  {merge_warning}\n"
        f"CLEANING_COMPLETE: {rows_merged} rows, {len(df_master.columns)} columns"
    )

    return summary


print("Tool registered:", store_clean_data.name)

Tool registered: store_clean_data


In [41]:
# ============================================================
# CELL 4f - Data Agent node assembly (fixed)
# ============================================================

from langchain_core.messages import SystemMessage
from langgraph.prebuilt import create_react_agent

# ---- Bind tools to the LLM ----
data_agent_tools = [load_raw_data, run_cleaning_code, store_clean_data]

# ---- Create the agent ----
# "prompt" replaces "state_modifier" in recent LangGraph versions
data_agent_runnable = create_react_agent(
    model=llm,
    tools=data_agent_tools,
    prompt=DATA_AGENT_PROMPT
)

# ---- Define the node function ----
def data_agent_node(state: PipelineState) -> dict:

    trigger = "Load both Excel files, audit them fully, clean them according \
to your standards, and store the final merged DataFrame."

    result = data_agent_runnable.invoke({
        "messages": [{"role": "user", "content": trigger}]
    })

    final_message = result["messages"][-1].content
    cleaned_json  = getattr(store_clean_data, "_cleaned_json",  "")
    cleaned_shape = getattr(store_clean_data, "_cleaned_shape", (0, 0))

    print(f"Data Agent finished.")
    print(f"Cleaned dataset shape: {cleaned_shape}")
    print(f"Final message preview:\n{final_message[:300]}")

    return {
        "messages":          result["messages"],
        "cleaned_data_json": cleaned_json,
    }

print("Data Agent node ready.")

Data Agent node ready.


C:\Users\gmitr\AppData\Local\Temp\ipykernel_8920\2924787887.py:13: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  data_agent_runnable = create_react_agent(


In [ ]:
# ============================================================
# CELL 4g - Test the Data Agent
# ============================================================

# Initialize a minimal state to trigger the agent
test_state = {
    "messages":          [],
    "cleaned_data_json": "",
    "baseline_stats":    "",
    "anomalies":         [],
    "confirmed_anomalies": [],
    "final_report":      "",
}

# Run the Data Agent node test
result = data_agent_node(test_state)

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.0-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 17.626715825s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url

In [43]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash-lite",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0,
)

response = llm.invoke("Say hello in one word.")
print("LLM test:", response.content)

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.0-flash-lite' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash-lite\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-lite\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-lite\nPlease retry in 9.927552252s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemin